# VRC-3PO camera-ready retraining

Use a GPU runtime: **Runtime → Change runtime type → T4 GPU**.

This notebook expects the archive itself, not a separate bundle. Either clone
the repository into Drive, or upload a ZIP of it when the next cell prompts:

```sh
git clone https://github.com/GokayToga/VRC-3PO.git
```

The harmonized dataset `vrc3po_master_dataset_fixed.csv` is **not** in the
archive and must be supplied separately — see *Data access* in the README.

Run the cells in order. The full run is resumable: rerunning the last cell
skips every completed protocol/model/seed combination.

In [ ]:
!pip install -q keras-tcn
import tensorflow as tf
import pandas as pd
import numpy as np
GPU_DEVICES = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPU devices:', GPU_DEVICES)
if not GPU_DEVICES:
    print('CPU mode is sufficient for the one-epoch smoke test. Use a GPU for the full run.')

In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile, shutil, os

drive.mount('/content/drive')
uploaded = files.upload()  # choose a ZIP of the VRC-3PO repository
zip_files = [name for name in uploaded if name.endswith('.zip')]
assert len(zip_files) == 1, f'Upload exactly one ZIP bundle; received: {list(uploaded)}'

bundle_root = Path('/content/vrc3po_camera_ready_bundle')
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir(parents=True)
with zipfile.ZipFile(zip_files[0]) as archive:
    archive.extractall(bundle_root)
os.chdir(bundle_root)
print('Bundle extracted to', bundle_root)
print('Analysis files:', sorted(str(p) for p in (bundle_root / 'analysis').glob('*.py')))

In [ ]:
from pathlib import Path

drive_root = Path('/content/drive/MyDrive')
direct = drive_root / 'vrc3po_master_dataset_fixed.csv'
if direct.exists():
    dataset_path = direct
else:
    matches = list(drive_root.rglob('vrc3po_master_dataset_fixed.csv'))
    assert matches, 'Dataset not found in MyDrive. Upload vrc3po_master_dataset_fixed.csv to MyDrive and rerun this cell.'
    dataset_path = matches[0]

output_dir = drive_root / 'vrc3po_camera_ready_results'
output_dir.mkdir(exist_ok=True)
DATASET = str(dataset_path)
OUTPUT = str(output_dir)
print('Dataset:', DATASET)
print('Output:', OUTPUT)
print('Dataset size (MB):', round(dataset_path.stat().st_size / 1e6, 1))

## Smoke test

This trains one MLP for one epoch. It should finish quickly and verifies that the data, imports, split logic, and output permissions all work.

In [ ]:
!python -m analysis.retrain_camera_ready --dataset "$DATASET" --output-dir "$OUTPUT/smoke_test" --variants MLP --protocols participant --seeds 42 --epochs 1
assert Path(OUTPUT, 'smoke_test', 'architecture_seed_metrics.csv').exists(), 'Smoke test did not create its metrics file.'
display(pd.read_csv(Path(OUTPUT, 'smoke_test', 'architecture_seed_metrics.csv')))
print('Smoke test passed.')

## Full retraining

This runs 7 architectures × 2 protocols × 3 seeds = 42 fits. Results are saved after every fit. If Colab disconnects or you stop the cell, reconnect, rerun the setup cells, and rerun this same cell; completed fits will show `[resume]` and will not be repeated.

In [ ]:
assert tf.config.list_physical_devices('GPU'), 'A GPU is required for the full 42-fit run. Reconnect when Colab GPU quota is available.'
!python -m analysis.retrain_camera_ready --dataset "$DATASET" --output-dir "$OUTPUT/full"

table4 = pd.read_csv(Path(OUTPUT, 'full', 'table4_protocol_comparison.csv'))
table5 = pd.read_csv(Path(OUTPUT, 'full', 'table5_architecture_ablation.csv'))
print('Table 4 source:')
display(table4)
print('Table 5 source:')
display(table5)
print('Finished. Results are in:', Path(OUTPUT, 'full'))

## Composition robustness (single-task binary endpoint)

The 42-fit run above uses one fixed participant assignment. This section asks
how much the single-task result depends on *which* people land in the test fold, by
repeating the whole pipeline over 20 fresh participant assignments.

Runs are saved one JSON per composition, so this cell is resumable in exactly
the same way as the full retraining cell.

**These compositions are not independent replicates.** They redraw from one
pool of 84 participants, so they share most of their training data. Report the
distribution; do not apply a one-sample test against 0.5.

In [ ]:
# Smoke test: 2 compositions, 3 epochs each. Should finish in a couple of minutes.
!python -m analysis.composition_robustness --dataset "$DATASET" --output-dir "$OUTPUT/composition_v2_smoke" --compositions 2 --epochs 3 --bootstrap 200
assert Path(OUTPUT, 'composition_smoke', 'composition_summary.json').exists(), 'Composition smoke test did not write its summary.'
display(pd.read_csv(Path(OUTPUT, 'composition_smoke', 'composition_results.csv')))
print('Composition smoke test passed.')


In [ ]:
# Full run: 20 compositions x 5 ensemble members = 100 fits.
# Safe to rerun; completed compositions print [resume] and are skipped.
!python -m analysis.composition_robustness --dataset "$DATASET" --output-dir "$OUTPUT/composition_v2" --compositions 20

import json
summary = json.loads(Path(OUTPUT, 'composition_v2', 'composition_summary.json').read_text())
results = pd.read_csv(Path(OUTPUT, 'composition_v2', 'composition_results.csv'))

print('Fill these into manuscript/ieee_access/pending_composition_subsection.tex:')
print(f"  <<N_COMPOSITIONS>>       {summary['n_compositions_attempted']}")
print(f"  <<N_COMPLETE>>           {summary['n_compositions_complete']}")
print(f"  <<MEAN_AUC>>             {summary['mean_auc']:.3f}")
print(f"  <<SD_AUC>>               {summary['sd_auc']:.3f}")
print(f"  <<MEDIAN_AUC>>           {summary['median_auc']:.3f}")
print(f"  <<MIN_AUC>>              {summary['min_auc']:.3f}")
print(f"  <<MAX_AUC>>              {summary['max_auc']:.3f}")
print(f"  <<N_CI_ABOVE>>           {summary['n_with_ci_above_chance']}")
print(f"  <<PERCENTILE_OF_FROZEN>> the {summary['frozen_split_percentile']:.0f}th percentile")
print(f"  <<COMPOSITION_RANGE>>    {summary['range_auc']:.3f}")
print()
print('Architecture span on the fixed split is 0.100. If range_auc above is not')
print('clearly larger, DROP the Discussion replacement paragraph at the bottom of')
print('the pending .tex and keep the hedged wording already in the manuscript.')
display(results)


In [ ]:
# Figure 7 for the manuscript. Writes into the manuscript figures folder.
!python -m analysis.generate_composition_figure --results-dir "$OUTPUT/composition_v2"
print('Copy fig7_composition.pdf/.png into manuscript/ieee_access/figures/ if you ran this outside the repo.')
